# Adım 3: Spark + Delta Lake — Bronze / Silver / Gold
**Kişi 2 sorumluluğu** — `feature/spark-eda` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month
import os

DATA_PATH   = './data/archive/daily_weather.parquet'
BRONZE_PATH = './delta_lake/bronze'
SILVER_PATH = './delta_lake/silver'
GOLD_PATH   = './delta_lake/gold'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateDataPipeline')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .config('spark.sql.parquet.datetimeRebaseModeInRead', 'CORRECTED')
        .config('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
        .getOrCreate()
    )

def write_bronze(spark):
    print('[BRONZE] Ham veri okunuyor...')
    df = spark.read.parquet(DATA_PATH)
    count = df.count()
    print(f'[BRONZE] Kayit sayisi: {count:,}')
    df.write.format('delta').mode('overwrite').save(BRONZE_PATH)
    print(f'[BRONZE] Yazildi -> {BRONZE_PATH}')
    return df

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
write_bronze(spark)

## Silver Katmanı
Null temizleme ve duplike kaldırma

In [ ]:
def write_silver(spark):
    print('[SILVER] Bronze okunuyor, temizleniyor...')
    df = spark.read.format('delta').load(BRONZE_PATH)
    df_clean = df.dropna(subset=['avg_temp_c', 'station_id', 'date'])
    before = df.count()
    df_clean = df_clean.dropDuplicates(['station_id', 'date'])
    after = df_clean.count()
    print(f'[SILVER] Ham: {before:,}  ->  Temiz: {after:,}  ({before - after:,} satir kaldirildi)')
    df_clean.write.format('delta').mode('overwrite').save(SILVER_PATH)
    print(f'[SILVER] Yazildi -> {SILVER_PATH}')
    return df_clean

write_silver(spark)

## Gold Katmanı
Fiziksel filtreler + year/month sütunları ekleniyor

In [ ]:
def write_gold(spark):
    print('[GOLD] Silver okunuyor, ozellikler ekleniyor...')
    df = spark.read.format('delta').load(SILVER_PATH)
    df_gold = (
        df
        .filter(col('avg_temp_c').between(-60, 60))
        .filter(col('min_temp_c') <= col('avg_temp_c'))
        .filter(col('avg_temp_c') <= col('max_temp_c'))
        .select(
            col('station_id'), col('city_name'), col('date'), col('season'),
            col('avg_temp_c'), col('min_temp_c'), col('max_temp_c'),
            col('precipitation_mm'), col('snow_depth_mm'),
            col('avg_wind_speed_kmh'), col('avg_sea_level_pres_hpa'),
            col('sunshine_total_min'),
            year(col('date')).alias('year'),
            month(col('date')).alias('month'),
        )
    )
    count = df_gold.count()
    print(f'[GOLD] Kayit sayisi: {count:,}')
    df_gold.write.format('delta').mode('overwrite').save(GOLD_PATH)
    print(f'[GOLD] Yazildi -> {GOLD_PATH}')
    return df_gold

df_gold = write_gold(spark)
print(f'Bronze -> {BRONZE_PATH}')
print(f'Silver -> {SILVER_PATH}')
print(f'Gold   -> {GOLD_PATH}')
df_gold.show(5, truncate=False)
spark.stop()
print('Pipeline tamamlandi.')